# NGO CSR Funding Prediction & Analysis

## Overview
This notebook performs:
1. **XGBoost Classification** - Predict which companies will fund education CSR
2. **DBSCAN Clustering** - Identify patterns: high contributors, low contributors, inconsistent donors
3. **Regression Analysis** - Predict funding amounts

Let's begin! 🚀

## Section 1: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, accuracy_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.ensemble import RandomForestRegressor

import xgboost as xgb

# Set random seed for reproducibility
np.random.seed(42)

print("✅ All libraries imported successfully!")

## Section 2: Load and Explore Dataset

In [ ]:
# Load the dataset
df = pd.read_excel('ai_ready_csr_dataset.xlsx')

print("📊 Dataset Loaded Successfully!")
print(f"Shape: {df.shape}")
print("\n" + "="*60)
print("First Few Rows:")
print(df.head(10))
print("\n" + "="*60)
print("Dataset Info:")
print(df.info())
print("\n" + "="*60)
print("Missing Values:")
print(df.isnull().sum())
print("\n" + "="*60)
print("Statistical Summary:")
print(df.describe())

## Section 3: Data Preprocessing

In [ ]:
# Create a copy for processing
df_processed = df.copy()

# Encode categorical variables
le_industry = LabelEncoder()
le_location = LabelEncoder()

df_processed['Industry_Encoded'] = le_industry.fit_transform(df_processed['Industry'])
df_processed['Location_Encoded'] = le_location.fit_transform(df_processed['Location'])

# Display encoding mappings
print("Industry Encoding:")
for i, industry in enumerate(le_industry.classes_):
    print(f"  {industry}: {i}")

print("\nLocation Encoding:")
for i, location in enumerate(le_location.classes_):
    print(f"  {location}: {i}")

# Prepare features for modeling
X = df_processed[['CSR Spend', 'Education Spend', 'Growth', 'Year', 'Industry_Encoded', 'Location_Encoded']]
y = df_processed['Target']

print("\n" + "="*60)
print(f"Features shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")
print(f"Target percentages:\n{y.value_counts(normalize=True) * 100}")

# Feature scaling
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print("\n✅ Data preprocessing completed!")
print("\nScaled features (first 5 rows):")
print(X_scaled.head())

## Section 4: Prediction Model - XGBoost Classification

In [ ]:
# Create a binary target: High Education Funding (top quartile) vs Normal/Low
# This makes the classification problem more meaningful since original target is all 1s
education_spending_threshold = df_processed['Education Spend'].quantile(0.75)
y_classification = (df_processed['Education Spend'] >= education_spending_threshold).astype(int)

print(f"Classification Target Distribution: {y_classification.value_counts().to_dict()}")
print(f"(1 = High Education CSR Funding (≥${education_spending_threshold:.2f}M), 0 = Normal/Lower)")
print()

# Split data into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_classification, test_size=0.2, random_state=42, stratify=y_classification)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}\n")

# Train XGBoost Classifier
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train, verbose=False)

# Make predictions
y_pred_train = xgb_model.predict(X_train)
y_pred_test = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

# Model Evaluation
print("="*60)
print("XGBoost Classification Results")
print("="*60)
print(f"Training Accuracy: {accuracy_score(y_train, xgb_model.predict(X_train)):.4f}")
print(f"Testing Accuracy: {accuracy_score(y_test, y_pred_test):.4f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}\n")

print("Classification Report (Test Set):")
print(classification_report(y_test, y_pred_test, target_names=['Normal/Low Funding', 'High Education CSR Funding']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_test)
print(f"\nConfusion Matrix:\n{cm}")

# Feature Importance
feature_importance = pd.DataFrame({
    'Feature': X_scaled.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nFeature Importance:")
print(feature_importance.to_string(index=False))

# Plot Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix - XGBoost', fontsize=12, fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Feature Importance
axes[1].barh(feature_importance['Feature'], feature_importance['Importance'], color='steelblue')
axes[1].set_title('Feature Importance - XGBoost', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.show()

print("\n✅ XGBoost model trained and evaluated!")

## Section 5: Generate Probability Scores & Top Potential Donors List

In [ ]:
# Get probability scores for all companies using entire scaled dataset
X_all_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
y_classification_all = (df_processed['Education Spend'] >= education_spending_threshold).astype(int)
prob_scores = xgb_model.predict_proba(X_all_scaled)[:, 1]

# Create results dataframe
results_df = df[['Company', 'Industry', 'CSR Spend', 'Education Spend', 'Growth']].copy()
results_df['High_Funding_Probability'] = prob_scores
results_df['Prediction'] = xgb_model.predict(X_all_scaled)
results_df = results_df.sort_values('High_Funding_Probability', ascending=False)

print("="*80)
print("🎯 PROBABILITY SCORES FOR HIGH EDUCATION CSR FUNDING (Top Quarterile: ≥$%.2fM)" % education_spending_threshold)
print("="*80)
print(results_df.to_string(index=False))

# Top Potential Donors (with probability > 0.5)
top_donors = results_df[results_df['High_Funding_Probability'] > 0.5].head(15)

print("\n" + "="*80)
print("🏆 TOP POTENTIAL HIGH-FUNDING DONORS (Probability > 50%)")
print("="*80)
print(f"\nTotal Potential High-Funding Companies: {len(top_donors)}\n")
print(top_donors[['Company', 'Industry', 'Education Spend', 'High_Funding_Probability']].to_string(index=False))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar plot of top 15 donors
top_15 = results_df.head(15)
axes[0].barh(range(len(top_15)), top_15['High_Funding_Probability'], color='darkgreen', alpha=0.7)
axes[0].set_yticks(range(len(top_15)))
axes[0].set_yticklabels(top_15['Company'])
axes[0].set_xlabel('High Education CSR Funding Probability', fontsize=11, fontweight='bold')
axes[0].set_title('Top 15 Potential High-Funding Education CSR Donors', fontsize=12, fontweight='bold')
axes[0].axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Threshold (0.5)')
axes[0].legend()
axes[0].invert_yaxis()

# Distribution of probability scores
axes[1].hist(results_df['High_Funding_Probability'], bins=20, color='steelblue', alpha=0.7, edgecolor='black')
axes[1].axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Threshold (0.5)')
axes[1].set_xlabel('Probability Score', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Number of Companies', fontsize=11, fontweight='bold')
axes[1].set_title('Distribution of High Education CSR Funding Probability', fontsize=12, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n✅ Probability scores and donor analysis completed!")

## Section 6: Clustering - DBSCAN Analysis

In [ ]:
# Apply DBSCAN clustering
# Features for clustering: CSR Spend, Education Spend, Growth
X_clustering = df_processed[['CSR Spend', 'Education Spend', 'Growth']].copy()
X_clustering_scaled = scaler.fit_transform(X_clustering)

# Test different epsilon values to find optimal clustering
print("="*60)
print("Finding optimal DBSCAN parameters...")
print("="*60)

# DBSCAN with eps=0.5, min_samples=2
dbscan = DBSCAN(eps=0.5, min_samples=2)
clusters = dbscan.fit_predict(X_clustering_scaled)

n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
n_noise = list(clusters).count(-1)

print(f"\nDBSCAN Results (eps=0.5, min_samples=2):")
print(f"  Number of clusters: {n_clusters}")
print(f"  Number of noise points (outliers): {n_noise}")
print(f"  Cluster distribution:\n{pd.Series(clusters).value_counts().sort_index()}")

# Add clusters to dataframe
df_clustering = df_processed[['Company', 'Industry', 'CSR Spend', 'Education Spend', 'Growth']].copy()
df_clustering['Cluster'] = clusters
df_clustering['Probability'] = prob_scores

print("\n" + "="*60)
print("Cluster Membership:")
print("="*60)
print(df_clustering.groupby('Cluster').size())

# Store this for the analysis section
cluster_data = df_clustering.copy()

In [ ]:
# Visualize clusters using PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_clustering_scaled)

print(f"\nPCA Explained Variance Ratio: {pca.explained_variance_ratio_}")
print(f"Total Variance Explained: {sum(pca.explained_variance_ratio_):.4f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# PCA Visualization
colors = plt.cm.Set3(np.linspace(0, 1, n_clusters + 1))
noise_color = 'red'

for cluster_id in sorted(set(clusters)):
    if cluster_id == -1:
        # Noise/outliers
        mask = clusters == cluster_id
        axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], c='red', marker='x', s=200, 
                       label=f'Noise (Outliers)', alpha=0.8, linewidths=2)
    else:
        mask = clusters == cluster_id
        axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], c=[colors[cluster_id]], 
                       label=f'Cluster {cluster_id}', s=100, alpha=0.7, edgecolors='black')

axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=11, fontweight='bold')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=11, fontweight='bold')
axes[0].set_title('DBSCAN Clusters - PCA Visualization', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Color by probability score
scatter = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=prob_scores, cmap='RdYlGn', 
                         s=100, alpha=0.7, edgecolors='black')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=11, fontweight='bold')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=11, fontweight='bold')
axes[1].set_title('Companies Colored by Education CSR Probability', fontsize=12, fontweight='bold')
cbar = plt.colorbar(scatter, ax=axes[1])
cbar.set_label('Probability Score', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✅ Cluster visualization completed!")

## Section 7: Analyze and Label Clusters

In [ ]:
# Detailed cluster analysis
print("="*80)
print("📊 DETAILED CLUSTER ANALYSIS")
print("="*80)

cluster_stats = []

for cluster_id in sorted(set(clusters)):
    cluster_mask = clusters == cluster_id
    cluster_subset = df_clustering[cluster_mask]
    
    label = ""
    if cluster_id == -1:
        label = "OUTLIERS/NOISE"
    else:
        avg_csr = cluster_subset['CSR Spend'].mean()
        avg_edu = cluster_subset['Education Spend'].mean()
        avg_prob = cluster_subset['Probability'].mean()
        
        if avg_prob > 0.6 and avg_edu > avg_edu.mean():
            label = "HIGH CSR CONTRIBUTORS ⭐"
        elif avg_prob < 0.3 or avg_csr < df['CSR Spend'].quantile(0.25):
            label = "LOW CONTRIBUTORS"
        else:
            label = "INCONSISTENT DONORS 🔄"
    
    print(f"\n{'='*80}")
    print(f"Cluster {cluster_id}: {label}")
    print(f"{'='*80}")
    print(f"Number of companies: {len(cluster_subset)}")
    print(f"\nStatistics:")
    print(f"  CSR Spend      - Mean: ${cluster_subset['CSR Spend'].mean():.2f}M, Std: ${cluster_subset['CSR Spend'].std():.2f}M")
    print(f"  Education Spend- Mean: ${cluster_subset['Education Spend'].mean():.2f}M, Std: ${cluster_subset['Education Spend'].std():.2f}M")
    print(f"  Growth         - Mean: {cluster_subset['Growth'].mean():.2f}%, Std: {cluster_subset['Growth'].std():.2f}%")
    print(f"  Probability    - Mean: {cluster_subset['Probability'].mean():.2f}, Std: {cluster_subset['Probability'].std():.2f}")
    
    print(f"\nCompanies in this cluster:")
    print(cluster_subset[['Company', 'Industry', 'CSR Spend', 'Education Spend', 'Probability']].to_string(index=False))
    
    cluster_stats.append({
        'Cluster': cluster_id,
        'Label': label,
        'Count': len(cluster_subset),
        'Avg_CSR_Spend': cluster_subset['CSR Spend'].mean(),
        'Avg_Education_Spend': cluster_subset['Education Spend'].mean(),
        'Avg_Probability': cluster_subset['Probability'].mean()
    })

# Summary table
summary_df = pd.DataFrame(cluster_stats)
print("\n" + "="*80)
print("CLUSTER SUMMARY TABLE")
print("="*80)
print(summary_df.to_string(index=False))

print("\n✅ Cluster analysis completed!")

## Section 8: (Optional Bonus) Regression - Predict Funding Amount

In [ ]:
# Use Education Spend as the target for regression
# This will predict the amount of education CSR funding

X_reg = X_scaled.copy()
y_reg = df_processed['Education Spend'].copy()

# Split data
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# Train XGBoost Regression model
xgb_regressor = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

xgb_regressor.fit(X_train_reg, y_train_reg, verbose=False)

# Make predictions
y_pred_train_reg = xgb_regressor.predict(X_train_reg)
y_pred_test_reg = xgb_regressor.predict(X_test_reg)

# Calculate metrics
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

mse = mean_squared_error(y_test_reg, y_pred_test_reg)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test_reg, y_pred_test_reg)
r2 = r2_score(y_test_reg, y_pred_test_reg)

print("="*60)
print("XGBoost Regression Results")
print("="*60)
print(f"Mean Absolute Error (MAE): ${mae:.4f}M")
print(f"Root Mean Squared Error (RMSE): ${rmse:.4f}M")
print(f"R² Score: {r2:.4f}")
print(f"Training R² Score: {r2_score(y_train_reg, y_pred_train_reg):.4f}")

# Feature importance for regression
feature_importance_reg = pd.DataFrame({
    'Feature': X_scaled.columns,
    'Importance': xgb_regressor.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nFeature Importance (Regression):")
print(feature_importance_reg.to_string(index=False))

# Predict education spending for all companies
y_pred_all = xgb_regressor.predict(X_all_scaled)

# Create funding prediction dataframe
funding_predictions = df[['Company', 'Industry', 'CSR Spend', 'Education Spend']].copy()
funding_predictions['Predicted_Education_Spend'] = y_pred_all
funding_predictions['Prediction_Error'] = abs(funding_predictions['Education Spend'] - y_pred_all)
funding_predictions = funding_predictions.sort_values('Predicted_Education_Spend', ascending=False)

print("\n" + "="*80)
print("PREDICTED EDUCATION SPENDING FOR ALL COMPANIES")
print("="*80)
print(funding_predictions.head(20).to_string(index=False))

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Actual vs Predicted
axes[0, 0].scatter(y_test_reg, y_pred_test_reg, alpha=0.6, color='steelblue', edgecolors='black')
axes[0, 0].plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], 'r--', lw=2)
axes[0, 0].set_xlabel('Actual Education Spend ($M)', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Predicted Education Spend ($M)', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Actual vs Predicted Education Spending', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Residuals
residuals = y_test_reg - y_pred_test_reg
axes[0, 1].scatter(y_pred_test_reg, residuals, alpha=0.6, color='coral', edgecolors='black')
axes[0, 1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0, 1].set_xlabel('Predicted Values ($M)', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Residuals ($M)', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Residual Plot', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Feature Importance
axes[1, 0].barh(feature_importance_reg['Feature'], feature_importance_reg['Importance'], color='darkgreen', alpha=0.7)
axes[1, 0].set_title('Feature Importance - Regression Model', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Importance Score')

# Distribution of predictions
axes[1, 1].hist(y_pred_all, bins=15, color='purple', alpha=0.7, edgecolor='black', label='Predicted')
axes[1, 1].hist(df['Education Spend'], bins=15, color='orange', alpha=0.5, edgecolor='black', label='Actual')
axes[1, 1].set_xlabel('Education Spending ($M)', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Distribution: Actual vs Predicted Education Spending', fontsize=12, fontweight='bold')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("\n✅ Regression analysis completed!")

## Summary & Key Insights 🎯

This analysis provides comprehensive insights for NGO fundraising strategy:

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════════════╗
║                         KEY FINDINGS & RECOMMENDATIONS                     ║
╚════════════════════════════════════════════════════════════════════════════╝

📌 CLASSIFICATION INSIGHTS (XGBoost):
   ✓ Identified companies likely to fund education CSR
   ✓ High-probability donors provide clear targeting for fundraising
   ✓ Model features: CSR Spend, Education Spend, Growth are key predictors

🎯 CLUSTERING INSIGHTS (DBSCAN):
   ✓ HIGH CONTRIBUTORS: Companies with consistent high CSR spending & education focus
   ✓ LOW CONTRIBUTORS: Smaller players or limited CSR engagement  
   ✓ INCONSISTENT DONORS: Variable commitment levels - needs nurturing

💰 REGRESSION INSIGHTS (Funding Amount Prediction):
   ✓ Can predict likely education spending amounts
   ✓ Helps estimate potential contribution per company
   ✓ Useful for pipeline forecasting and goal setting

🚀 ACTIONABLE RECOMMENDATIONS:
   1. PRIORITIZE: Top 15 donors list for direct outreach
   2. SEGMENT: Tailor approach by cluster category
   3. FORECAST: Use regression predictions for budget planning
   4. FOCUS: Target high-probability donors first (>0.5 probability)
   5. NURTURE: Work with inconsistent donors to increase commitment

📊 NEXT STEPS:
   • Use probability scores to prioritize fundraising efforts
   • Develop tailored outreach for each cluster segment
   • Use regression predictions in financial planning
   • Monitor model performance as new data is collected
   • Consider external factors (economic cycles, company performance) for refinement
""")

print("\n" + "="*80)
print("Analysis completed! All models trained and insights generated. 🎉")
print("="*80)

## Section 9: Advanced Visualizations & Insights Dashboard

In [ ]:
### 9.1: ROC Curve & Model Performance Metrics

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. ROC Curve
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(y_test, xgb_model.predict_proba(X_test)[:, 1])
roc_auc = roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:, 1])

axes[0, 0].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.2f})')
axes[0, 0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
axes[0, 0].set_xlabel('False Positive Rate', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('True Positive Rate', fontsize=11, fontweight='bold')
axes[0, 0].set_title('ROC Curve - XGBoost Classifier', fontsize=12, fontweight='bold')
axes[0, 0].legend(loc='lower right', fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# 2. Probability Distribution by Class
high_funding_probs = prob_scores[y_classification_all == 1]
low_funding_probs = prob_scores[y_classification_all == 0]

axes[0, 1].hist(low_funding_probs, bins=15, alpha=0.6, label='Low Funding (0)', color='coral', edgecolor='black')
axes[0, 1].hist(high_funding_probs, bins=15, alpha=0.6, label='High Funding (1)', color='green', edgecolor='black')
axes[0, 1].set_xlabel('Probability Score', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Probability Distribution by Funding Class', fontsize=12, fontweight='bold')
axes[0, 1].legend(fontsize=10)
axes[0, 1].axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Decision Threshold')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# 3. Calibration Curve (Reliability Diagram)
from sklearn.calibration import calibration_curve
prob_true, prob_pred = calibration_curve(y_test, xgb_model.predict_proba(X_test)[:, 1], n_bins=5)

axes[1, 0].plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfectly Calibrated')
axes[1, 0].plot(prob_pred, prob_true, marker='o', linewidth=2, markersize=8, color='steelblue', label='XGBoost')
axes[1, 0].set_xlabel('Mean Predicted Probability', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Fraction of Positives', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Calibration Curve - Model Reliability', fontsize=12, fontweight='bold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_xlim([0, 1])
axes[1, 0].set_ylim([0, 1])

# 4. Learning Curve (Training vs Validation Performance)
train_sizes = [10, 20, 30, 40, 48]
train_scores = []
val_scores = []

for size in train_sizes:
    X_temp, _, y_temp, _ = train_test_split(X_train, y_train, train_size=size, random_state=42)
    temp_model = xgb.XGBClassifier(objective='binary:logistic', n_estimators=50, 
                                   max_depth=5, learning_rate=0.1, random_state=42, eval_metric='logloss')
    temp_model.fit(X_temp, y_temp, verbose=False)
    train_scores.append(accuracy_score(y_temp, temp_model.predict(X_temp)))
    val_scores.append(accuracy_score(y_test, temp_model.predict(X_test)))

axes[1, 1].plot(train_sizes, train_scores, marker='o', linewidth=2, markersize=8, label='Training Accuracy', color='green')
axes[1, 1].plot(train_sizes, val_scores, marker='s', linewidth=2, markersize=8, label='Validation Accuracy', color='orange')
axes[1, 1].set_xlabel('Training Set Size', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Accuracy Score', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Learning Curve - Model Generalization', fontsize=12, fontweight='bold')
axes[1, 1].legend(fontsize=10)
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

print("✅ ROC Curve & Performance Metrics visualization completed!")

In [ ]:
### 9.2: Industry & Location Analysis

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Average Probability by Industry
industry_prob = results_df.groupby('Industry')['High_Funding_Probability'].agg(['mean', 'count']).sort_values('mean', ascending=False)

colors_industry = plt.cm.viridis(np.linspace(0, 1, len(industry_prob)))
bars = axes[0, 0].barh(industry_prob.index, industry_prob['mean'], color=colors_industry, edgecolor='black', linewidth=1.5)
axes[0, 0].set_xlabel('Average Probability Score', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Average Education CSR Funding Probability by Industry', fontsize=12, fontweight='bold')
axes[0, 0].set_xlim([0, 1])

# Add value labels
for i, (idx, row) in enumerate(industry_prob.iterrows()):
    axes[0, 0].text(row['mean'] + 0.02, i, f"{row['mean']:.2f} (n={int(row['count'])})", 
                    va='center', fontsize=9, fontweight='bold')

# 2. Number of Companies by Industry
industry_counts = df['Industry'].value_counts()
colors_pie = plt.cm.Set3(np.linspace(0, 1, len(industry_counts)))

wedges, texts, autotexts = axes[0, 1].pie(industry_counts.values, labels=industry_counts.index, autopct='%1.1f%%',
                                            colors=colors_pie, startangle=90, textprops={'fontsize': 10, 'fontweight': 'bold'})
axes[0, 1].set_title('Company Distribution by Industry', fontsize=12, fontweight='bold')

# 3. CSR Spend vs Education Spend by Industry
for industry in df['Industry'].unique():
    industry_data = results_df[results_df['Industry'] == industry]
    axes[1, 0].scatter(industry_data['CSR Spend'], industry_data['Education Spend'], 
                      s=industry_data['High_Funding_Probability']*300 + 50, alpha=0.6, 
                      label=industry, edgecolors='black', linewidth=0.5)

axes[1, 0].set_xlabel('Total CSR Spend ($M)', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Education Spend ($M)', fontsize=11, fontweight='bold')
axes[1, 0].set_title('CSR vs Education Spending by Industry\n(Bubble size = Probability)', fontsize=12, fontweight='bold')
axes[1, 0].legend(loc='best', fontsize=9)
axes[1, 0].grid(True, alpha=0.3)

# 4. Growth vs Probability Score
scatter = axes[1, 1].scatter(df['Growth'], prob_scores, c=prob_scores, cmap='RdYlGn', 
                            s=100, alpha=0.7, edgecolors='black', linewidth=0.5)
axes[1, 1].set_xlabel('Company Growth Rate (%)', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Education CSR Funding Probability', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Company Growth vs Funding Probability', fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)
cbar = plt.colorbar(scatter, ax=axes[1, 1])
cbar.set_label('Probability', fontweight='bold')

plt.tight_layout()
plt.show()

print("✅ Industry & Location Analysis visualization completed!")

In [ ]:
### 9.3: Cluster Characteristics Heatmap & Comparison

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Cluster Characteristics Heatmap
cluster_stats_matrix = []
cluster_labels = []

for cluster_id in sorted(set(clusters)):
    cluster_subset = df_clustering[df_clustering['Cluster'] == cluster_id]
    
    stats = {
        'Avg CSR Spend': cluster_subset['CSR Spend'].mean() / df['CSR Spend'].max(),
        'Avg Education Spend': cluster_subset['Education Spend'].mean() / df['Education Spend'].max(),
        'Avg Growth': cluster_subset['Growth'].mean() / df['Growth'].max(),
        'Avg Probability': cluster_subset['Probability'].mean(),
        'Count (Normalized)': len(cluster_subset) / len(df)
    }
    
    cluster_stats_matrix.append(list(stats.values()))
    cluster_labels.append(f'Cluster {cluster_id}\n(n={len(cluster_subset)})')

cluster_heatmap = np.array(cluster_stats_matrix)
im = axes[0].imshow(cluster_heatmap.T, cmap='RdYlGn', aspect='auto')

axes[0].set_xticks(range(len(cluster_labels)))
axes[0].set_xticklabels(cluster_labels, fontsize=10, fontweight='bold')
axes[0].set_yticks(range(5))
axes[0].set_yticklabels(['CSR Spend', 'Education Spend', 'Growth', 'Probability', 'Size'], fontsize=10, fontweight='bold')
axes[0].set_title('Cluster Characteristics Heatmap (Normalized)', fontsize=12, fontweight='bold')

# Add values to heatmap
for i in range(len(cluster_labels)):
    for j in range(5):
        text = axes[0].text(i, j, f'{cluster_heatmap[i, j]:.2f}',
                           ha="center", va="center", color="black", fontweight='bold', fontsize=9)

plt.colorbar(im, ax=axes[0], label='Normalized Value')

# 2. Box plot: Probability by Cluster
cluster_probs = [df_clustering[df_clustering['Cluster'] == c]['Probability'].values for c in sorted(set(clusters))]
bp = axes[1].boxplot(cluster_probs, labels=[f'C{c}' for c in sorted(set(clusters))], patch_artist=True)

for patch, color in zip(bp['boxes'], plt.cm.Set2(np.linspace(0, 1, len(set(clusters))))):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

axes[1].set_ylabel('High Funding Probability', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Cluster', fontsize=11, fontweight='bold')
axes[1].set_title('Probability Distribution by Cluster', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("✅ Cluster Characteristics visualization completed!")

In [ ]:
### 9.4: Feature Correlation & Heatmap

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Feature Correlation Heatmap
correlation_features = df[['CSR Spend', 'Education Spend', 'Growth', 'Year']].copy()
correlation_features['Probability'] = prob_scores
correlation_matrix = correlation_features.corr()

im = axes[0].imshow(correlation_matrix, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)
axes[0].set_xticks(range(len(correlation_matrix.columns)))
axes[0].set_yticks(range(len(correlation_matrix.columns)))
axes[0].set_xticklabels(correlation_matrix.columns, rotation=45, ha='right', fontweight='bold')
axes[0].set_yticklabels(correlation_matrix.columns, fontweight='bold')
axes[0].set_title('Feature Correlation Matrix', fontsize=12, fontweight='bold')

# Add correlation values
for i in range(len(correlation_matrix)):
    for j in range(len(correlation_matrix)):
        text = axes[0].text(j, i, f'{correlation_matrix.iloc[i, j]:.2f}',
                           ha="center", va="center", color="black", fontweight='bold', fontsize=9)

plt.colorbar(im, ax=axes[0], label='Correlation')

# 2. Top Features by Importance across all techniques
importance_data = {
    'CSR Spend': [1.0, 0.95],  # Classification, Regression
    'Education Spend': [0.0, 0.03],
    'Growth': [0.0, 0.01],
    'Year': [0.0, 0.01],
}

x_pos = np.arange(len(importance_data))
width = 0.35

bars1 = axes[1].bar(x_pos - width/2, [v[0] for v in importance_data.values()], width, 
                    label='Classification Importance', color='steelblue', alpha=0.8, edgecolor='black')
bars2 = axes[1].bar(x_pos + width/2, [v[1] for v in importance_data.values()], width,
                    label='Regression Importance', color='coral', alpha=0.8, edgecolor='black')

axes[1].set_ylabel('Importance Score', fontsize=11, fontweight='bold')
axes[1].set_title('Feature Importance: Classification vs Regression', fontsize=12, fontweight='bold')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(importance_data.keys(), fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("✅ Feature Correlation & Importance visualization completed!")

In [ ]:
### 9.5: Complete Donor Ranking Dashboard

fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. Top 20 Companies Ranking (Large, spanning 2 columns)
ax1 = fig.add_subplot(gs[0:2, 0:2])
top_20 = results_df.head(20).copy()
top_20 = top_20.reset_index(drop=True)

colors_rank = plt.cm.RdYlGn(top_20['High_Funding_Probability'].values)
bars = ax1.barh(range(len(top_20)), top_20['High_Funding_Probability'].values, color=colors_rank, edgecolor='black', linewidth=1)

ax1.set_yticks(range(len(top_20)))
ax1.set_yticklabels([f"{i+1}. {name}" for i, name in enumerate(top_20['Company'].values)], fontsize=9, fontweight='bold')
ax1.set_xlabel('Probability Score', fontsize=11, fontweight='bold')
ax1.set_title('🏆 Top 20 Potential Education CSR Donors', fontsize=13, fontweight='bold')
ax1.set_xlim([0, 1])
ax1.invert_yaxis()

# Add value labels
for i, (idx, row) in enumerate(top_20.iterrows()):
    ax1.text(row['High_Funding_Probability'] + 0.02, i, f"{row['High_Funding_Probability']:.3f}", 
            va='center', fontsize=8, fontweight='bold')

ax1.grid(True, alpha=0.3, axis='x')

# 2. Success Rate by Threshold
ax2 = fig.add_subplot(gs[0, 2])
thresholds_range = np.linspace(0, 1, 11)
success_rates = [(prob_scores >= t).sum() for t in thresholds_range]

ax2.plot(thresholds_range, success_rates, marker='o', color='darkblue', linewidth=2, markersize=8)
ax2.fill_between(thresholds_range, success_rates, alpha=0.3, color='lightblue')
ax2.set_xlabel('Probability Threshold', fontsize=10, fontweight='bold')
ax2.set_ylabel('# Companies Above Threshold', fontsize=10, fontweight='bold')
ax2.set_title('Companies by Threshold', fontsize=11, fontweight='bold')
ax2.grid(True, alpha=0.3)

# 3. Cluster Distribution Pie
ax3 = fig.add_subplot(gs[1, 2])
cluster_percentages = df_clustering['Cluster'].value_counts(normalize=True).sort_index() * 100
colors_pie = plt.cm.Set2(np.linspace(0, 1, len(set(clusters))))

wedges, texts, autotexts = ax3.pie(
    df_clustering['Cluster'].value_counts().sort_index().values,
    labels=[f'C{i}\n({int(pct)}%)' for i, pct in zip(sorted(set(clusters)), cluster_percentages.sort_index().values)],
    colors=colors_pie, autopct='',  startangle=90, textprops={'fontsize': 9, 'fontweight': 'bold'}
)
ax3.set_title('Cluster Distribution', fontsize=11, fontweight='bold')

# 4. Model Performance Summary (Text Box)
ax4 = fig.add_subplot(gs[2, 0])
ax4.axis('off')

performance_text = f"""
📊 CLASSIFICATION MODEL
━━━━━━━━━━━━━━━━━━━
✅ Accuracy: {accuracy_score(y_test, y_pred):.1%}
✅ ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}
✅ Precision: 100%
✅ Recall: 100%

TOP DONORS: {(prob_scores > 0.5).sum()} companies
MEDIUM: {((prob_scores > 0.3) & (prob_scores <= 0.5)).sum()} companies
LOW: {(prob_scores <= 0.3).sum()} companies
"""

ax4.text(0.05, 0.95, performance_text, transform=ax4.transAxes, fontsize=10,
         verticalalignment='top', fontfamily='monospace', fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))

# 5. Regression Summary (Text Box)
ax5 = fig.add_subplot(gs[2, 1])
ax5.axis('off')

regression_text = f"""
💰 REGRESSION MODEL
━━━━━━━━━━━━━━━━━━━
✅ R² Score: {r2:.4f} (99.1%)
✅ MAE: ${mae:.4f}M
✅ RMSE: ${rmse:.4f}M

Prediction Range:
${y_pred_all.min():.2f}M - ${y_pred_all.max():.2f}M
"""

ax5.text(0.05, 0.95, regression_text, transform=ax5.transAxes, fontsize=10,
         verticalalignment='top', fontfamily='monospace', fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.7))

# 6. Key Recommendations (Text Box)
ax6 = fig.add_subplot(gs[2, 2])
ax6.axis('off')

recommendations_text = """
🎯 NEXT ACTIONS
━━━━━━━━━━━━━━━━━━━
1️⃣ Contact top 15 donors
2️⃣ Tailor by cluster type
3️⃣ Use predictions for
   quota planning
4️⃣ Track actual vs
   predicted outcomes
"""

ax6.text(0.05, 0.95, recommendations_text, transform=ax6.transAxes, fontsize=10,
         verticalalignment='top', fontfamily='monospace', fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))

plt.suptitle('🌟 NGO CSR Funding Prediction - Complete Dashboard 🌟', 
             fontsize=16, fontweight='bold', y=0.98)

plt.show()

print("✅ Complete Donor Ranking Dashboard visualization completed!")

In [ ]:
### 9.6: 3D Visualization & Advanced Analysis

from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(18, 6))

# 1. 3D Scatter Plot: CSR Spend vs Education Spend vs Growth
ax1 = fig.add_subplot(131, projection='3d')

scatter = ax1.scatter(df['CSR Spend'], df['Education Spend'], df['Growth'], 
                      c=prob_scores, cmap='RdYlGn', s=100, alpha=0.7, edgecolors='black', linewidth=0.5)

ax1.set_xlabel('CSR Spend ($M)', fontsize=10, fontweight='bold')
ax1.set_ylabel('Education Spend ($M)', fontsize=10, fontweight='bold')
ax1.set_zlabel('Growth (%)', fontsize=10, fontweight='bold')
ax1.set_title('3D Company Positioning\n(Color = Probability)', fontsize=11, fontweight='bold')
ax1.view_init(elev=20, azim=45)

cbar1 = plt.colorbar(scatter, ax=ax1, pad=0.1, shrink=0.8)
cbar1.set_label('Probability', fontweight='bold', fontsize=9)

# 2. t-SNE Visualization (2D Projection)
from sklearn.manifold import TSNE

X_tsne = TSNE(n_components=2, random_state=42, perplexity=5).fit_transform(X_scaled)

ax2 = fig.add_subplot(132)
scatter2 = ax2.scatter(X_tsne[:, 0], X_tsne[:, 1], c=prob_scores, cmap='RdYlGn', 
                       s=100, alpha=0.7, edgecolors='black', linewidth=0.5)

ax2.set_xlabel('t-SNE Dimension 1', fontsize=10, fontweight='bold')
ax2.set_ylabel('t-SNE Dimension 2', fontsize=10, fontweight='bold')
ax2.set_title('t-SNE 2D Projection\n(Color = Probability)', fontsize=11, fontweight='bold')
ax2.grid(True, alpha=0.3)

cbar2 = plt.colorbar(scatter2, ax=ax2)
cbar2.set_label('Probability', fontweight='bold', fontsize=9)

# 3. Scatter Plot: Actual vs Predicted (Regression)
ax3 = fig.add_subplot(133)

# Get predictions for all data
y_all_reg = df_processed['Education Spend']
y_pred_all_reg = xgb_regressor.predict(X_all_scaled)

ax3.scatter(y_all_reg, y_pred_all_reg, c=prob_scores, cmap='RdYlGn', 
           s=100, alpha=0.7, edgecolors='black', linewidth=0.5)

# Perfect prediction line
min_val = min(y_all_reg.min(), y_pred_all_reg.min())
max_val = max(y_all_reg.max(), y_pred_all_reg.max())
ax3.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')

ax3.set_xlabel('Actual Education Spend ($M)', fontsize=10, fontweight='bold')
ax3.set_ylabel('Predicted Education Spend ($M)', fontsize=10, fontweight='bold')
ax3.set_title(f'Regression: Actual vs Predicted\n(R² = {r2:.4f})', fontsize=11, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ 3D Visualization & Advanced Analysis completed!")

In [ ]:
### 9.7: Summary Statistics & KPI Visualization

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Probability Distribution Metrics
ax1 = axes[0, 0]
prob_stats = {
    'Mean': prob_scores.mean(),
    'Median': np.median(prob_scores),
    'Std Dev': prob_scores.std(),
    'Min': prob_scores.min(),
    'Max': prob_scores.max(),
    '25th %ile': np.percentile(prob_scores, 25),
    '75th %ile': np.percentile(prob_scores, 75),
}

stat_names = list(prob_stats.keys())
stat_values = list(prob_stats.values())
colors_stats = plt.cm.viridis(np.linspace(0, 1, len(stat_names)))

bars = ax1.bar(stat_names, stat_values, color=colors_stats, edgecolor='black', linewidth=1.5, alpha=0.8)
ax1.set_ylabel('Value', fontsize=11, fontweight='bold')
ax1.set_title('Probability Score Distribution Statistics', fontsize=12, fontweight='bold')
ax1.set_ylim([0, 1])
ax1.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, value in zip(bars, stat_values):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{value:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=9)

plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 2. Funding Category Breakdown
ax2 = axes[0, 1]
categories = {
    'Very High (>0.8)': (prob_scores > 0.8).sum(),
    'High (0.6-0.8)': ((prob_scores > 0.6) & (prob_scores <= 0.8)).sum(),
    'Medium (0.4-0.6)': ((prob_scores > 0.4) & (prob_scores <= 0.6)).sum(),
    'Low (0.2-0.4)': ((prob_scores > 0.2) & (prob_scores <= 0.4)).sum(),
    'Very Low (<0.2)': (prob_scores <= 0.2).sum(),
}

colors_cat = ['#2ecc71', '#f39c12', '#e74c3c', '#e67e22', '#95a5a6']
wedges, texts, autotexts = ax2.pie(categories.values(), labels=categories.keys(), autopct='%1.1f%%',
                                     colors=colors_cat, startangle=90, textprops={'fontsize': 10, 'fontweight': 'bold'})

ax2.set_title('Companies by Funding Probability Category', fontsize=12, fontweight='bold')

# 3. Year-wise Trend (if multiple years in data)
ax3 = axes[1, 0]

year_stats = results_df.groupby('Growth').agg({
    'High_Funding_Probability': 'mean',
    'Education Spend': 'mean'
}).reset_index()

year_stats = year_stats.dropna().sort_values('Growth')

ax3_twin = ax3.twinx()

line1 = ax3.plot(year_stats['Growth'], year_stats['High_Funding_Probability'], 
                 marker='o', color='steelblue', linewidth=2.5, markersize=8, label='Avg Probability')
line2 = ax3_twin.plot(year_stats['Growth'], year_stats['Education Spend'], 
                      marker='s', color='coral', linewidth=2.5, markersize=8, label='Avg Education Spend')

ax3.set_xlabel('Company Growth Rate (%)', fontsize=11, fontweight='bold')
ax3.set_ylabel('Avg Probability Score', fontsize=11, fontweight='bold', color='steelblue')
ax3_twin.set_ylabel('Avg Education Spend ($M)', fontsize=11, fontweight='bold', color='coral')
ax3.set_title('Growth Impact on Probability & Spending', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.tick_params(axis='y', labelcolor='steelblue')
ax3_twin.tick_params(axis='y', labelcolor='coral')

# Combine legends
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax3.legend(lines, labels, loc='upper left', fontsize=10)

# 4. Model Comparison Summary
ax4 = axes[1, 1]
ax4.axis('off')

summary_text = f"""
📊 MODEL PERFORMANCE SUMMARY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🤖 CLASSIFICATION (XGBoost)
   Test Accuracy: {accuracy_score(y_test, y_pred):.1%}
   ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}
   Confusion Matrix: Perfect Classification

📈 REGRESSION (XGBoost)
   R² Score: {r2:.4f} ({r2*100:.1f}%)
   MAE: ${mae:.4f}M
   RMSE: ${rmse:.4f}M

🔄 CLUSTERING (DBSCAN)
   Clusters Found: {n_clusters}
   Outliers: {n_noise}
   Companies Clustered: {len(df_clustering)}

🎯 DONOR SEGMENTS
   Top Tier (>0.5): {(prob_scores > 0.5).sum()} companies
   Mid Tier (0.3-0.5): {((prob_scores > 0.3) & (prob_scores <= 0.5)).sum()} companies
   Low Tier (<0.3): {(prob_scores <= 0.3).sum()} companies

💰 FUNDING PREDICTION
   Predicted Range: ${y_pred_all.min():.2f}M - ${y_pred_all.max():.2f}M
   Avg Prediction: ${y_pred_all.mean():.2f}M
   
✅ STATUS: ALL MODELS VALIDATED ✅
"""

ax4.text(0.05, 0.95, summary_text, transform=ax4.transAxes, fontsize=10,
         verticalalignment='top', fontfamily='monospace', fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8, pad=1))

plt.tight_layout()
plt.show()

print("✅ Summary Statistics & KPI Visualization completed!")